In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-14B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

try:
    # rindex finding 151668
    index = len(output_ids) - output_ids[::-1].index(151668) # "</think>"
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

In [ ]:
tools = [
    {
        "name": "get_weather",
        "description": "Get the weather in a city",
        "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "The city to get the weather for"}}}}
]

messages = [
    {
        "role": "user",
        "content": "Hello! How is the weather today in Copenhagen?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    enable_thinking=True,
    tools=tools,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
)
outputs = model.generate(inputs.to(model.device), max_new_tokens=32768)

In [ ]:
import yaml

with open("./config/qwen-sft.yaml") as f:
    data = yaml.safe_load(f)

In [ ]:
from datasets import load_dataset

ds = load_dataset("eungizoa/agentic-collection", name="SFT", split="hermes_function_calling_v1_no_think")

In [ ]:
import json
from datasets import Features, Sequence, Value, Dataset, DatasetDict
from typing import List, Dict, Any

def compose_messages_with_thinking(messages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Recompose parsed messages into the serialized conversation format:
    
    [
      {"role": "user", "content": "~~"},
      {"role": "assistant", "content": "<think>...</think>\n<tool_call>...</tool_call>"},
      {"role": "tool", "content": "~~"},
      {"role": "assistant", "content": "~~"},
      ...
    ]

    Rules:
      - Assistant messages may contain:
          * reasoning_content  → wrapped in <think>...</think>
          * tool_calls          → each call wrapped separately in <tool_call>...</tool_call>
          * content            → plain text (public output)
      - Consecutive assistant messages (reasoning/tool/content) are merged into one.
      - Tool and user messages remain as-is.
    """
    composed: List[Dict[str, Any]] = []
    buffer: Dict[str, Any] = None  # holds partial assistant message

    def flush_buffer():
        """If an assistant message is being built, finalize and append it."""
        nonlocal buffer
        if buffer is not None:
            content_parts = []

            # Handle reasoning section
            if "reasoning_content" in buffer and buffer["reasoning_content"]:
                content_parts.append(f"<think>\n{buffer['reasoning_content'].strip()}\n</think>")
            else:
                content_parts.append(f"<think>\n\n</think>")

            # Handle tool calls
            if "tool_calls" in buffer and buffer["tool_calls"]:
                tool_calls = buffer["tool_calls"]

                # If it's a list, serialize each individually
                if isinstance(tool_calls, list):
                    for tool_call in tool_calls:
                        tool_str = json.dumps(tool_call, ensure_ascii=False)
                        content_parts.append(f"<tool_call>\n{tool_str}\n</tool_call>")
                # Single tool_call (dict)
                elif isinstance(tool_calls, dict):
                    tool_str = json.dumps(tool_calls, ensure_ascii=False)
                    content_parts.append(f"<tool_call>\n{tool_str}\n</tool_call>")
                # Fallback: raw string
                else:
                    content_parts.append(f"<tool_call>\n{str(tool_calls)}\n</tool_call>")

            # Handle normal assistant text
            if "content" in buffer and buffer["content"]:
                content_parts.append(buffer["content"].strip())

            # Combine all parts
            final_content = "\n".join(content_parts).strip()
            composed.append({"role": "assistant", "content": final_content})
            buffer = None

    # Iterate through messages
    for msg in messages:
        role = msg.get("role")
        if role == "assistant":
            # Start or continue assistant message
            if buffer is None:
                buffer = {"role": "assistant"}

            # Merge fields accordingly
            if "reasoning_content" in msg:
                buffer["reasoning_content"] = (
                    buffer.get("reasoning_content", "") + "\n" + msg["reasoning_content"]
                ).strip()
            elif "tool_calls" in msg:
                buffer["tool_calls"] = msg["tool_calls"]
            elif "content" in msg:
                buffer["content"] = (
                    buffer.get("content", "") + "\n" + msg["content"]
                ).strip()
        else:
            flush_buffer()
            composed.append(msg)

    # Flush any remaining assistant block
    flush_buffer()
    return composed

def format_messages(example: dict):
    example["messages"] = json.loads(example["messages"])
    example["messages"] = compose_messages_with_thinking(example["messages"])
    return example

ds = ds.map(format_messages, features=None)

In [ ]:
for example in ds:
    example["tools"] = json.loads(example["tools"])

In [ ]:
from datasets import load_dataset

ds = load_dataset("eungizoa/agentic-collection", name="SFT", split="smolagents_toolcalling_traces")

In [ ]:
import json
from typing import List, Dict, Any

def compose_messages_with_thinking(messages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Recompose parsed messages into the serialized conversation format:
    
    [
      {"role": "user", "content": "~~"},
      {"role": "assistant", "content": "<think>...</think>\n<tool_call>...</tool_call>"},
      {"role": "tool", "content": "~~"},
      {"role": "assistant", "content": "~~"},
      ...
    ]

    Rules:
      - Assistant messages may contain:
          * reasoning_content  → wrapped in <think>...</think>
          * tool_calls          → each call wrapped separately in <tool_call>...</tool_call>
          * content            → plain text (public output)
      - Consecutive assistant messages (reasoning/tool/content) are merged into one.
      - Tool and user messages remain as-is.
    """
    composed: List[Dict[str, Any]] = []
    buffer: Dict[str, Any] = None  # holds partial assistant message

    def flush_buffer():
        """If an assistant message is being built, finalize and append it."""
        nonlocal buffer
        if buffer is not None:
            content_parts = []

            # Handle reasoning section
            if "reasoning_content" in buffer and buffer["reasoning_content"]:
                content_parts.append(f"<think>\n{buffer['reasoning_content'].strip()}\n</think>")
            else:
                content_parts.append(f"<think>\n\n</think>")

            # Handle tool calls
            if "tool_calls" in buffer and buffer["tool_calls"]:
                tool_calls = buffer["tool_calls"]

                # If it's a list, serialize each individually
                if isinstance(tool_calls, list):
                    for tool_call in tool_calls:
                        tool_str = json.dumps(tool_call, ensure_ascii=False)
                        content_parts.append(f"<tool_call>\n{tool_str}\n</tool_call>")
                # Single tool_call (dict)
                elif isinstance(tool_calls, dict):
                    tool_str = json.dumps(tool_calls, ensure_ascii=False)
                    content_parts.append(f"<tool_call>\n{tool_str}\n</tool_call>")
                # Fallback: raw string
                else:
                    content_parts.append(f"<tool_call>\n{str(tool_calls)}\n</tool_call>")

            # Handle normal assistant text
            if "content" in buffer and buffer["content"]:
                content_parts.append(buffer["content"].strip())

            # Combine all parts
            final_content = "\n".join(content_parts).strip()
            composed.append({"role": "assistant", "content": final_content})
            buffer = None

    # Iterate through messages
    for msg in messages:
        role = msg.get("role")
        if role == "assistant":
            # Start or continue assistant message
            if buffer is None:
                buffer = {"role": "assistant"}

            # Merge fields accordingly
            if "reasoning_content" in msg:
                buffer["reasoning_content"] = (
                    buffer.get("reasoning_content", "") + "\n" + msg["reasoning_content"]
                ).strip()
            elif "tool_calls" in msg:
                buffer["tool_calls"] = msg["tool_calls"]
            elif "content" in msg:
                buffer["content"] = (
                    buffer.get("content", "") + "\n" + msg["content"]
                ).strip()
        else:
            flush_buffer()
            composed.append(msg)

    # Flush any remaining assistant block
    flush_buffer()
    return composed

example["messages"] = compose_messages_with_thinking(example["messages"])

In [ ]:
example["messages"]

In [ ]:
print(tokenizer.apply_chat_template(
    example["messages"][:2],
    tools=example["tools"],
    tokenize=False,
    enable_thinking=True,
))

## Test Qwen request

In [ ]:
messages = [
  {
    "content": "I've recently installed a new security system at my home, and I want to ensure everything is functioning as it should. Specifically, I'd like to start by checking the live feed from the camera located at the front door to monitor any activity. The camera has a unique identifier, which I've already configured to be \"front_door.\" I'd prefer to view the live stream in high definition, so a 1080p quality would be ideal. Could you please call the appropriate function to retrieve the live feed from my front door camera in 1080p quality and provide me with the link to the stream?\n\nFollowing this, I would also like to record the live feed from this camera for the next 30 minutes. This is to test the recording feature and to keep an archived copy for security purposes. Please initiate the recording function for the \"front_door\" camera with a recording duration of 30 minutes.\n\nLastly, as part of my routine surveillance checks, I need to review footage from yesterday between 3 PM and 5 PM. The camera \"front_garden\" should have the recording for that period. I need to retrieve this specific recorded feed. The start time for the recording was at 15:00 on April 22, 2023, and it ended at 17:00 on the same day.\n\nTo summarize, I request the execution of the following functions with the provided details:\n1. Retrieve the live feed from the \"front_door\" camera at 1080p quality.\n2. Start a 30-minute recording of the live feed from the \"front_door\" camera.\n3. Retrieve the recorded feed from the \"front_garden\" camera, focusing on the time period between 15:00 and 17:00 on April 22, 2023.\n\nThank you for assisting with the management of my home security camera feeds.",
    "role": "user"
  }
]

tools = [
  {
    "type": "function",
    "function": {
      "name": "get_camera_live_feed",
      "description": "Retrieves the live feed from a specified security camera.",
      "parameters": {
        "type": "object",
        "properties": {
          "camera_id": {
            "type": "string",
            "description": "The unique identifier for the camera."
          },
          "stream_quality": {
            "type": "string",
            "description": "The desired quality of the live stream.",
            "enum": [
              "720p",
              "1080p",
              "4k"
            ]
          }
        },
        "required": [
          "camera_id"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "list_all_cameras",
      "description": "Lists all the security cameras connected to the home network.",
      "parameters": {
        "type": "object",
        "properties": {
          "include_offline": {
            "type": "boolean",
            "description": "Whether to include cameras that are currently offline.",
            "default": False
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "record_camera_feed",
      "description": "Starts recording the live feed from a specified security camera.",
      "parameters": {
        "type": "object",
        "properties": {
          "camera_id": {
            "type": "string",
            "description": "The unique identifier for the camera."
          },
          "duration": {
            "type": "integer",
            "description": "The duration in minutes for which to record the feed.",
            "default": 60
          }
        },
        "required": [
          "camera_id"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_recorded_feed",
      "description": "Retrieves a previously recorded feed from a specified security camera.",
      "parameters": {
        "type": "object",
        "properties": {
          "camera_id": {
            "type": "string",
            "description": "The unique identifier for the camera."
          },
          "start_time": {
            "type": "string",
            "description": "The start time of the recording to retrieve, in ISO 8601 format."
          },
          "end_time": {
            "type": "string",
            "description": "The end time of the recording to retrieve, in ISO 8601 format."
          }
        },
        "required": [
          "camera_id",
          "start_time",
          "end_time"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "pan_tilt_camera",
      "description": "Controls the pan and tilt functions of a PTZ (Pan-Tilt-Zoom) security camera.",
      "parameters": {
        "type": "object",
        "properties": {
          "camera_id": {
            "type": "string",
            "description": "The unique identifier for the PTZ camera."
          },
          "pan_angle": {
            "type": "integer",
            "description": "The angle in degrees to pan the camera. Positive values pan right, negative values pan left."
          },
          "tilt_angle": {
            "type": "integer",
            "description": "The angle in degrees to tilt the camera. Positive values tilt up, negative values tilt down."
          }
        },
        "required": [
          "camera_id",
          "pan_angle",
          "tilt_angle"
        ]
      }
    }
  }
]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-14B")

In [ ]:
print(tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
))

In [ ]:
import requests
import json

def query_local_completion(prompt: str):
    url = "http://localhost:8006/v1/completions"
    headers = {"Content-Type": "application/json"}
    data = {
        "model": "Qwen/Qwen3-14B",
        "prompt": prompt,
        "max_tokens": 2048,
        "temperature": 0
    }

    response = requests.post(url, headers=headers, data=json.dumps(data))

    # Raise error if request failed
    response.raise_for_status()

    result = response.json()
    # Extract and return the completion text
    return result["choices"][0]["text"]

prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)

response = query_local_completion(prompt)

In [ ]:
response

In [ ]:
print(response)

In [ ]:
messages = [
  {
    "content": "I've recently installed a new security system at my home, and I want to ensure everything is functioning as it should. Specifically, I'd like to start by checking the live feed from the camera located at the front door to monitor any activity. The camera has a unique identifier, which I've already configured to be \"front_door.\" I'd prefer to view the live stream in high definition, so a 1080p quality would be ideal. Could you please call the appropriate function to retrieve the live feed from my front door camera in 1080p quality and provide me with the link to the stream?\n\nFollowing this, I would also like to record the live feed from this camera for the next 30 minutes. This is to test the recording feature and to keep an archived copy for security purposes. Please initiate the recording function for the \"front_door\" camera with a recording duration of 30 minutes.\n\nLastly, as part of my routine surveillance checks, I need to review footage from yesterday between 3 PM and 5 PM. The camera \"front_garden\" should have the recording for that period. I need to retrieve this specific recorded feed. The start time for the recording was at 15:00 on April 22, 2023, and it ended at 17:00 on the same day.\n\nTo summarize, I request the execution of the following functions with the provided details:\n1. Retrieve the live feed from the \"front_door\" camera at 1080p quality.\n2. Start a 30-minute recording of the live feed from the \"front_door\" camera.\n3. Retrieve the recorded feed from the \"front_garden\" camera, focusing on the time period between 15:00 and 17:00 on April 22, 2023.\n\nThank you for assisting with the management of my home security camera feeds.",
    "role": "user"
  },
  {
    "role": "assistant",
    "content": response,  
  },
  {
    "role": "tool",
    "content": "{\"name\": \"get_camera_live_feed\", \"content\": {\"camera_id\": \"front_door\", \"stream_quality\": \"1080p\", \"live_feed_url\": \"https://homecam.example.com/live/front_door_1080p\"}}"
  },
  {
    "role": "tool",
    "content": "{\"name\": \"record_camera_feed\", \"content\": {\"camera_id\": \"front_door\", \"duration\": 30, \"recording_status\": \"started\", \"recording_url\": \"https://homecam.example.com/recordings/front_door_20230423T143000Z.mp4\"}}"
  },
  {
    "role": "tool",
    "content": "{\"name\": \"get_recorded_feed\", \"content\": {\"camera_id\": \"front_garden\", \"start_time\": \"2023-04-22T15:00:00Z\", \"end_time\": \"2023-04-22T17:00:00Z\", \"recorded_feed_url\": \"https://homecam.example.com/recordings/front_garden_20230422T150000Z_to_20230422T170000Z.mp4\"}}"
  }
]
prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    enable_thinking=False,
    add_generation_prompt=True
)

In [ ]:
response = query_local_completion(prompt)

In [ ]:
print(response)

In [ ]:
from rich import print
from rich.text import Text

# Visualize using rich

dicts = {'messages': [{'content': 'New task:\nA vessel of volume $22.4 \\mathrm{dm}^3$ contains $2.0 \\mathrm{~mol} \\mathrm{H}_2$ and $1.0 \\mathrm{~mol} \\mathrm{~N}_2$ at $273.15 \\mathrm{~K}$. Calculate their total pressure. Answer with only the final number in $\\mathrm{atm}$  unit.', 'role': 'user'}, {'content': '<think>\nI have already calculated the total number of moles (2.0 mol H₂ + 1.0 mol N₂ = 3.0 mol), and since the conditions are STP with a 22.4 dm³ vessel, using the ideal gas law, the total pressure is 3.0 atm for 3 moles of gas in 22.4 dm³ at 273.15 K.\n</think>\n<tool_call>\n{"name": "final_answer", "arguments": {"answer": "3.0"}}\n</tool_call>', 'role': 'assistant'}], 'chat_template_kwargs': {'enable_thinking': True}, 'tools': [{'function': {'description': 'Provides a final answer to the given problem.', 'name': 'final_answer', 'parameters': {'properties': {'answer': {'description': 'The final answer to the problem', 'type': 'string'}}, 'required': ['answer'], 'type': 'object'}}, 'type': 'function'}, {'function': {'description': 'Performs a web search for a query and returns a string of the top search results formatted as markdown with titles, links, and descriptions.', 'name': 'web_search', 'parameters': {'properties': {'query': {'description': 'The search query to perform.', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}, 'type': 'function'}, {'function': {'description': 'Searches Wikipedia and returns a summary or full text of the given topic, along with the page URL.', 'name': 'wikipedia_search', 'parameters': {'properties': {'query': {'description': 'The topic to search on Wikipedia.', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}, 'type': 'function'}], 'input_ids': [151644, 8948, 198, 2, 13852, 271, 2610, 1231, 1618, 825, 476, 803, 5746, 311, 7789, 448, 279, 1196, 3239, 382, 2610, 525, 3897, 448, 729, 32628, 2878, 366, 15918, 1472, 15918, 29, 11874, 9492, 510, 27, 15918, 397, 4913, 1688, 788, 5212, 4684, 788, 330, 49022, 264, 1590, 4226, 311, 279, 2661, 3491, 10465, 330, 606, 788, 330, 11822, 28534, 497, 330, 13786, 788, 5212, 13193, 788, 5212, 9217, 788, 5212, 4684, 788, 330, 785, 1590, 4226, 311, 279, 3491, 497, 330, 1313, 788, 330, 917, 9207, 2137, 330, 6279, 788, 4383, 9217, 7914, 330, 1313, 788, 330, 1700, 9207, 2137, 330, 1313, 788, 330, 1688, 16707, 4913, 1688, 788, 5212, 4684, 788, 330, 3889, 9807, 264, 3482, 2711, 369, 264, 3239, 323, 4675, 264, 914, 315, 279, 1909, 2711, 3059, 23126, 438, 50494, 448, 15311, 11, 7746, 11, 323, 27787, 10465, 330, 606, 788, 330, 2911, 10716, 497, 330, 13786, 788, 5212, 13193, 788, 5212, 1631, 788, 5212, 4684, 788, 330, 785, 2711, 3239, 311, 2736, 10465, 330, 1313, 788, 330, 917, 9207, 2137, 330, 6279, 788, 4383, 1631, 7914, 330, 1313, 788, 330, 1700, 9207, 2137, 330, 1313, 788, 330, 1688, 16707, 4913, 1688, 788, 5212, 4684, 788, 330, 5890, 288, 26587, 323, 4675, 264, 12126, 476, 2480, 1467, 315, 279, 2661, 8544, 11, 3156, 448, 279, 2150, 5548, 10465, 330, 606, 788, 330, 86, 14939, 10716, 497, 330, 13786, 788, 5212, 13193, 788, 5212, 1631, 788, 5212, 4684, 788, 330, 785, 8544, 311, 2711, 389, 26587, 10465, 330, 1313, 788, 330, 917, 9207, 2137, 330, 6279, 788, 4383, 1631, 7914, 330, 1313, 788, 330, 1700, 9207, 2137, 330, 1313, 788, 330, 1688, 16707, 522, 15918, 1339, 2461, 1817, 729, 1618, 11, 470, 264, 2951, 1633, 448, 729, 829, 323, 5977, 2878, 220, 151657, 151658, 11874, 9492, 510, 151657, 198, 4913, 606, 788, 366, 1688, 11494, 8066, 330, 16370, 788, 366, 2116, 56080, 40432, 31296, 151658, 151645, 198, 151644, 872, 198, 3564, 3383, 510, 32, 26182, 315, 8123, 400, 17, 17, 13, 19, 1124, 91550, 90, 13849, 92, 61, 18, 3, 5610, 400, 17, 13, 15, 1124, 91550, 90, 93, 44344, 92, 1124, 91550, 90, 39, 19444, 17, 3, 323, 400, 16, 13, 15, 1124, 91550, 90, 93, 44344, 92, 1124, 91550, 90, 93, 45, 19444, 17, 3, 518, 400, 17, 22, 18, 13, 16, 20, 1124, 91550, 90, 93, 42, 92, 12947, 20517, 862, 2790, 7262, 13, 21806, 448, 1172, 279, 1590, 1372, 304, 57960, 91550, 90, 266, 76, 31716, 220, 4982, 13, 151645, 198, 151644, 77091, 198, 151667, 198, 40, 614, 2669, 16588, 279, 2790, 1372, 315, 4544, 642, 320, 17, 13, 15, 21609, 472, 31807, 488, 220, 16, 13, 15, 21609, 451, 31807, 284, 220, 18, 13, 15, 21609, 701, 323, 2474, 279, 4682, 525, 3928, 47, 448, 264, 220, 17, 17, 13, 19, 28676, 43201, 26182, 11, 1667, 279, 10507, 6819, 2329, 11, 279, 2790, 7262, 374, 220, 18, 13, 15, 69787, 369, 220, 18, 4544, 642, 315, 6819, 304, 220, 17, 17, 13, 19, 28676, 43201, 518, 220, 17, 22, 18, 13, 16, 20, 730, 624, 151668, 271, 151657, 198, 4913, 606, 788, 330, 11822, 28534, 497, 330, 16370, 788, 5212, 9217, 788, 330, 18, 13, 15, 95642, 151658, 151645, 198], 'assistant_masks': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


text_visualization = Text()
tokens = tokenizer.convert_ids_to_tokens(dicts["input_ids"])
for token, mask in zip(tokens, dicts["assistant_masks"]):
    color = "cyan" if mask else "white"
    text_visualization.append(token.replace('Ġ', ' ').replace("Ċ", "\n"), style=color)

print(text_visualization)